# ⛰️ Heaps & Priority Queues — Runnable Notebook

Companion to [`../tutorials/12_Heaps_Priority_Queues.md`](../tutorials/12_Heaps_Priority_Queues.md) and
[`../html/12_heaps_priority_queues.html`](../html/12_heaps_priority_queues.html).

A **min-heap** built from scratch (sift up/down), then Python's `heapq` and two classic uses.

## 1. A min-heap from scratch
Stored as a plain list; `left=2i+1`, `right=2i+2`, `parent=(i-1)//2`.

In [ ]:
def push(heap, x):
    """Add x at the end, then SIFT UP while it is smaller than its parent."""
    heap.append(x)
    i = len(heap) - 1
    while i > 0:
        parent = (i - 1) // 2
        if heap[i] < heap[parent]:              # smaller than parent -> bubble up
            heap[i], heap[parent] = heap[parent], heap[i]
            i = parent
        else:
            break                               # heap property restored

def pop(heap):
    """Remove the root (minimum). Move the last item up, then SIFT DOWN."""
    top = heap[0]
    last = heap.pop()
    if heap:
        heap[0] = last
        i, n = 0, len(heap)
        while True:
            small, l, r = i, 2*i + 1, 2*i + 2
            if l < n and heap[l] < heap[small]: small = l
            if r < n and heap[r] < heap[small]: small = r
            if small == i:                      # no smaller child -> done
                break
            heap[i], heap[small] = heap[small], heap[i]
            i = small
    return top

h = []
for x in [5, 1, 8, 3, 9, 2, 7]:
    push(h, x)
print("heap array (root is the min):", h)
print("root / peek:", h[0])
assert h[0] == 1                                 # the minimum is always at index 0

# popping repeatedly yields sorted order (this is heap-sort)
out = [pop(h) for _ in range(7)]
print("popped in order:", out)
assert out == sorted(out) == [1, 2, 3, 5, 7, 8, 9]

## 2. heapify — turn any array into a heap in O(n)

In [ ]:
def sift_down(a, i, n):
    while True:
        small, l, r = i, 2*i + 1, 2*i + 2
        if l < n and a[l] < a[small]: small = l
        if r < n and a[r] < a[small]: small = r
        if small == i:
            return
        a[i], a[small] = a[small], a[i]
        i = small

def heapify(a):
    """Bottom-up: sift-down every internal node. Total work is O(n), not O(n log n)."""
    n = len(a)
    for i in range(n // 2 - 1, -1, -1):
        sift_down(a, i, n)

arr = [9, 4, 7, 1, 3, 8, 2]
heapify(arr)
print("heapified:", arr)
assert arr[0] == min(arr)                        # root is the minimum

## 3. Python's `heapq` (min-heap) and the max-heap trick

In [ ]:
import heapq

h = []
for x in [5, 1, 3]:
    heapq.heappush(h, x)
print("heappop ->", heapq.heappop(h), "(the minimum)")

# order by an explicit key using tuples (priority, item) -- as Dijkstra does
pq = []
heapq.heappush(pq, (2, "task-B"))
heapq.heappush(pq, (1, "task-A"))
print("highest priority:", heapq.heappop(pq))    # (1, 'task-A')

# MAX-heap: negate on the way in and out
mx = []
for x in [5, 1, 3]:
    heapq.heappush(mx, -x)            # push negatives
largest = -heapq.heappop(mx)          # negate on the way out -> the maximum
print("max via negation:", largest)
assert largest == 5

## 4. Top-K with a size-K min-heap — O(n log k)

In [ ]:
import heapq

def top_k(nums, k):
    """Keep a min-heap of size k; the k largest survive."""
    h = []
    for x in nums:
        heapq.heappush(h, x)
        if len(h) > k:
            heapq.heappop(h)                     # drop the smallest -> only top-k remain
    return sorted(h, reverse=True)

print("top 3 of [4,1,7,3,9,2,8]:", top_k([4, 1, 7, 3, 9, 2, 8], 3))
assert top_k([4, 1, 7, 3, 9, 2, 8], 3) == [9, 8, 7]

## 5. Streaming median with two heaps — O(1) median
A max-heap for the lower half, a min-heap for the upper half, kept balanced.

In [ ]:
import heapq

class MedianFinder:
    def __init__(self):
        self.lo = []          # max-heap (store negatives) -> lower half
        self.hi = []          # min-heap -> upper half

    def add(self, x):
        heapq.heappush(self.lo, -x)                        # tentatively to lower half
        heapq.heappush(self.hi, -heapq.heappop(self.lo))   # move its max to upper half
        if len(self.hi) > len(self.lo):                    # rebalance so lo >= hi in size
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def median(self):
        if len(self.lo) > len(self.hi):
            return -self.lo[0]                             # odd count -> middle of lower half
        return (-self.lo[0] + self.hi[0]) / 2              # even count -> average of middles

mf = MedianFinder()
for x in [1, 2, 3]:
    mf.add(x)
print("median of 1,2,3 :", mf.median())
mf.add(4)
print("median of 1,2,3,4:", mf.median())
assert mf.median() == 2.5

## ✅ Recap
- A heap is a **complete binary tree** in an array; **root = min** (or max).
- **push** = append + sift-up; **pop** = swap-last-to-root + sift-down; both `O(log n)`, peek `O(1)`.
- **heapify** builds a heap in **`O(n)`**.
- `heapq` is min-only (negate for max); store `(priority, item)` tuples.
- Powers **top-K**, **two-heap median**, Dijkstra/Prim, and schedulers.

Next: [`13_Tries`](../tutorials/13_Tries.md).